# Residual Diagnostics

After fitting a forecasting model, the critical question is: **has the model
captured all the signal in the data, or is there useful information left in
the residuals?**

## Definitions

- **Fitted value** $\hat{y}_t$: the one-step-ahead forecast for observation
  $t$, computed using only data available *before* time $t$.
- **Residual** $e_t = y_t - \hat{y}_t$: the forecast error at time $t$.

## Properties of Good Residuals

1. **Uncorrelated** — the ACF should lie within the significance bounds.
   If residuals are correlated, there is information left to exploit.
2. **Zero mean** — if the mean is non-zero, the forecasts are biased.
3. **Constant variance** (homoscedastic) — variance should not change with
   the level or over time.
4. **Normally distributed** — nice to have, needed for valid prediction
   intervals, but not strictly required.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

In [ ]:
# Load airline passengers
airline = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    index_col="Month",
    parse_dates=True,
)
airline.index.freq = "MS"
airline.columns = ["Passengers"]
passengers = airline["Passengers"]

print(f"Shape: {airline.shape}")
airline.head()

---
## 1. Computing Residuals for the Naive Method

For the naive method, $\hat{y}_t = y_{t-1}$, so the residuals are simply
the first differences:

$$
e_t = y_t - y_{t-1}
$$

In [ ]:
# Fitted values for naive: y_{t-1}
fitted_naive = passengers.shift(1)

# Residuals
residuals_naive = passengers - fitted_naive
residuals_naive = residuals_naive.dropna()

print(f"Number of residuals: {len(residuals_naive)}")
print(f"Mean of residuals: {residuals_naive.mean():.4f}")
print(f"Std of residuals:  {residuals_naive.std():.4f}")
residuals_naive.head(10)

In [ ]:
# Visual check: overlay fitted and actual
fig, ax = plt.subplots()
ax.plot(passengers, label="Actual", color="tab:blue")
ax.plot(fitted_naive, label="Fitted (naive)", color="tab:red", alpha=0.7, linestyle="--")
ax.set_title("Naive Method — Actual vs Fitted")
ax.set_ylabel("Thousands of Passengers")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()

---
## 2. Four Diagnostic Plots

The standard residual diagnostics consist of four plots:

1. **Time plot** of residuals — look for patterns, changing variance
2. **ACF plot** — look for significant autocorrelations
3. **Histogram** — check if distribution is roughly symmetric / normal
4. **Q-Q plot** — compare residual quantiles to theoretical normal quantiles

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Time plot
axes[0, 0].plot(residuals_naive)
axes[0, 0].axhline(0, color="grey", linestyle="--", alpha=0.5)
axes[0, 0].set_title("Residuals over Time")
axes[0, 0].set_ylabel("Residual")

# 2. ACF
plot_acf(residuals_naive, ax=axes[0, 1], lags=36, alpha=0.05)
axes[0, 1].set_title("ACF of Residuals")

# 3. Histogram
axes[1, 0].hist(residuals_naive, bins=20, edgecolor="black", density=True, alpha=0.7)
# Overlay normal curve
x = np.linspace(residuals_naive.min(), residuals_naive.max(), 100)
axes[1, 0].plot(x, stats.norm.pdf(x, residuals_naive.mean(), residuals_naive.std()),
                color="tab:red", linewidth=2, label="Normal fit")
axes[1, 0].set_title("Histogram of Residuals")
axes[1, 0].legend()

# 4. Q-Q plot
stats.probplot(residuals_naive, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title("Q-Q Plot")

plt.suptitle("Residual Diagnostics — Naive Method on Airline Passengers", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### Interpreting the Plots

- **Time plot**: the variance clearly *increases* over time — the residuals
  are heteroscedastic.  This makes sense because airline passengers grow,
  and a naive forecast's errors grow with the level.
- **ACF**: multiple significant spikes, especially at lag 12 — there is
  strong **seasonal autocorrelation** left in the residuals.  The naive
  method has not captured the seasonal pattern.
- **Histogram**: roughly bell-shaped but with heavier tails than normal.
- **Q-Q plot**: deviations in the tails confirm non-normality.

---
## 3. Reusable `check_residuals()` Function

Let us package the four-plot diagnostic into a reusable function that can
be applied to any model's residuals.

In [ ]:
def check_residuals(residuals, lags=36, title="Residual Diagnostics"):
    """
    Produce the four standard residual diagnostic plots and print
    summary statistics.

    Parameters
    ----------
    residuals : pd.Series
        Model residuals (y - y_hat), with NaNs already dropped.
    lags : int
        Number of lags for the ACF plot.
    title : str
        Suptitle for the figure.
    """
    resid = residuals.dropna()

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 1. Time plot
    axes[0, 0].plot(resid)
    axes[0, 0].axhline(0, color="grey", linestyle="--", alpha=0.5)
    axes[0, 0].set_title("Residuals over Time")
    axes[0, 0].set_ylabel("Residual")

    # 2. ACF
    plot_acf(resid, ax=axes[0, 1], lags=min(lags, len(resid) // 2 - 1), alpha=0.05)
    axes[0, 1].set_title("ACF of Residuals")

    # 3. Histogram with normal overlay
    axes[1, 0].hist(resid, bins=20, edgecolor="black", density=True, alpha=0.7)
    x = np.linspace(resid.min(), resid.max(), 100)
    axes[1, 0].plot(x, stats.norm.pdf(x, resid.mean(), resid.std()),
                    color="tab:red", linewidth=2, label="Normal fit")
    axes[1, 0].set_title("Histogram of Residuals")
    axes[1, 0].legend()

    # 4. Q-Q plot
    stats.probplot(resid, dist="norm", plot=axes[1, 1])
    axes[1, 1].set_title("Q-Q Plot")

    plt.suptitle(title, fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

    # Summary statistics
    print(f"Mean:     {resid.mean():.4f}")
    print(f"Std:      {resid.std():.4f}")
    print(f"Skewness: {resid.skew():.4f}")
    print(f"Kurtosis: {resid.kurtosis():.4f}")


print("check_residuals() function defined.")

In [ ]:
# Test the function on our naive residuals
check_residuals(residuals_naive, title="Naive Method — Airline Passengers")

---
## 4. Residuals from Other Methods

Let us compare residual diagnostics across different simple methods.

In [ ]:
# Mean method: fitted value at time t is the expanding mean up to t-1
fitted_mean = passengers.expanding().mean().shift(1)
residuals_mean = (passengers - fitted_mean).dropna()

check_residuals(residuals_mean, title="Mean Method — Airline Passengers")

In [ ]:
# Seasonal naive: fitted value at time t is y_{t-12}
fitted_snaive = passengers.shift(12)
residuals_snaive = (passengers - fitted_snaive).dropna()

check_residuals(residuals_snaive, title="Seasonal Naive — Airline Passengers")

### Comparison

- **Mean method**: massive autocorrelation at all lags — it has captured
  neither the trend nor the seasonality.
- **Naive method**: strong autocorrelation at seasonal lags — it misses
  the seasonality.
- **Seasonal naive**: better, but still shows some trend-related structure
  because the mean of the residuals is positive (growing trend means
  next year's value is typically higher than this year's).

---
## 5. The Ljung-Box Test

The **Ljung-Box test** is a formal statistical test for whether
autocorrelations in the residuals are collectively zero.  It tests:

- $H_0$: The residuals are independently distributed (no autocorrelation)
- $H_1$: The residuals exhibit serial correlation

The test statistic is:

$$
Q^* = T(T+2) \sum_{k=1}^{K} \frac{r_k^2}{T-k}
$$

where $r_k$ is the autocorrelation at lag $k$ and $K$ is the number of
lags tested.  Under $H_0$, $Q^* \sim \chi^2(K)$.

**Rule of thumb**: use $K = 10$ for non-seasonal data, $K = 2m$ for
seasonal data (where $m$ is the seasonal period).

In [ ]:
# Ljung-Box test on naive residuals
lb_naive = acorr_ljungbox(residuals_naive, lags=[12, 24], return_df=True)
print("Ljung-Box Test — Naive Method Residuals")
print("="*50)
print(lb_naive)
print()
if (lb_naive["lb_pvalue"] < 0.05).any():
    print("CONCLUSION: p-value < 0.05 — reject H0.")
    print("Residuals have significant autocorrelation.")
    print("The naive method has NOT captured all available information.")
else:
    print("CONCLUSION: p-value >= 0.05 — fail to reject H0.")
    print("No evidence of remaining autocorrelation.")

In [ ]:
# Ljung-Box test on seasonal naive residuals
lb_snaive = acorr_ljungbox(residuals_snaive, lags=[12, 24], return_df=True)
print("Ljung-Box Test — Seasonal Naive Residuals")
print("="*50)
print(lb_snaive)
print()
if (lb_snaive["lb_pvalue"] < 0.05).any():
    print("CONCLUSION: Significant autocorrelation remains.")
else:
    print("CONCLUSION: No significant autocorrelation detected.")

In [ ]:
# Ljung-Box at multiple lags to see the pattern
lb_detailed = acorr_ljungbox(residuals_naive, lags=range(1, 25), return_df=True)

fig, ax = plt.subplots()
ax.bar(lb_detailed.index, -np.log10(lb_detailed["lb_pvalue"]), color="tab:blue", alpha=0.7)
ax.axhline(-np.log10(0.05), color="tab:red", linestyle="--", label="p = 0.05 threshold")
ax.set_xlabel("Lag")
ax.set_ylabel("$-\\log_{10}(p)$ (higher = more significant)")
ax.set_title("Ljung-Box p-values — Naive Residuals")
ax.legend()
plt.tight_layout()
plt.show()

print("Bars above the red line indicate significant autocorrelation.")
print("The naive residuals are very significantly autocorrelated at all lags tested.")

---
## 6. Checking for Bias (Non-Zero Mean)

If the mean of the residuals is significantly different from zero, the
forecasts are **biased** — they systematically over- or under-predict.

In [ ]:
# One-sample t-test: is the mean of residuals significantly different from 0?
def test_bias(residuals, name=""):
    """Test if residual mean is significantly different from zero."""
    t_stat, p_value = stats.ttest_1samp(residuals.dropna(), 0)
    mean_val = residuals.mean()
    print(f"{name}")
    print(f"  Mean residual: {mean_val:.4f}")
    print(f"  t-statistic:   {t_stat:.4f}")
    print(f"  p-value:       {p_value:.4f}")
    if p_value < 0.05:
        direction = "over-predicting" if mean_val < 0 else "under-predicting"
        print(f"  BIASED — forecasts are systematically {direction}")
    else:
        print(f"  UNBIASED — no evidence of systematic bias")
    print()


test_bias(residuals_naive, "Naive Method")
test_bias(residuals_snaive, "Seasonal Naive")
test_bias(residuals_mean, "Mean Method")

---
## 7. Checking for Heteroscedasticity

Residuals with **changing variance** indicate that the model performs
differently at different levels of the series.  This often signals a need
for a variance-stabilising transformation (e.g., log transform).

In [ ]:
# Split residuals into halves and compare variance
n = len(residuals_naive)
first_half = residuals_naive.iloc[:n//2]
second_half = residuals_naive.iloc[n//2:]

print("Variance comparison (naive residuals):")
print(f"  First half variance:  {first_half.var():.2f}")
print(f"  Second half variance: {second_half.var():.2f}")
print(f"  Ratio: {second_half.var() / first_half.var():.2f}")
print()

# Levene's test for equal variances
stat, p = stats.levene(first_half, second_half)
print(f"Levene's test: statistic={stat:.4f}, p-value={p:.4f}")
if p < 0.05:
    print("Reject H0: variances are significantly different (heteroscedastic).")
else:
    print("Fail to reject H0: no evidence of unequal variances.")

In [ ]:
# Visualise: rolling std of residuals
rolling_std = residuals_naive.rolling(12).std()

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(residuals_naive)
axes[0].axhline(0, color="grey", linestyle="--", alpha=0.5)
axes[0].set_title("Naive Residuals")
axes[0].set_ylabel("Residual")

axes[1].plot(rolling_std, color="tab:orange")
axes[1].set_title("Rolling Standard Deviation (window=12)")
axes[1].set_ylabel("Std")
axes[1].set_xlabel("Date")

plt.tight_layout()
plt.show()

print("The rolling std is clearly increasing — residuals are heteroscedastic.")
print("A log transformation of the original data would help stabilise the variance.")

---
## 8. Normality Tests

While normality is the least critical property (the first two — uncorrelated
and zero mean — are essential), it is needed for valid prediction intervals.

Common tests:
- **Shapiro-Wilk** test (best for small to medium samples)
- **Jarque-Bera** test (based on skewness and kurtosis)

In [ ]:
# Shapiro-Wilk test
sw_stat, sw_p = stats.shapiro(residuals_naive)
print(f"Shapiro-Wilk test: statistic={sw_stat:.4f}, p-value={sw_p:.4f}")
if sw_p < 0.05:
    print("Reject H0: residuals are NOT normally distributed.")
else:
    print("Fail to reject H0: no evidence against normality.")
print()

# Jarque-Bera test
jb_stat, jb_p = stats.jarque_bera(residuals_naive)
print(f"Jarque-Bera test: statistic={jb_stat:.4f}, p-value={jb_p:.4f}")
if jb_p < 0.05:
    print("Reject H0: residuals are NOT normally distributed.")
else:
    print("Fail to reject H0: no evidence against normality.")

---
## 9. Complete Diagnostic Report

Let us create an enhanced `check_residuals()` that includes all tests.

In [ ]:
def full_residual_check(residuals, lags=36, lb_lags=None, title="Residual Diagnostics"):
    """
    Complete residual diagnostic: four plots + statistical tests.

    Parameters
    ----------
    residuals : pd.Series
        Model residuals.
    lags : int
        Number of lags for ACF plot.
    lb_lags : list of int, optional
        Lags for Ljung-Box test. Default [12, 24].
    title : str
        Plot title.
    """
    resid = residuals.dropna()
    if lb_lags is None:
        lb_lags = [12, 24]

    # --- Plots ---
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].plot(resid)
    axes[0, 0].axhline(0, color="grey", linestyle="--", alpha=0.5)
    axes[0, 0].set_title("Residuals over Time")
    axes[0, 0].set_ylabel("Residual")

    plot_acf(resid, ax=axes[0, 1], lags=min(lags, len(resid) // 2 - 1), alpha=0.05)
    axes[0, 1].set_title("ACF of Residuals")

    axes[1, 0].hist(resid, bins=20, edgecolor="black", density=True, alpha=0.7)
    x = np.linspace(resid.min(), resid.max(), 100)
    axes[1, 0].plot(x, stats.norm.pdf(x, resid.mean(), resid.std()),
                    color="tab:red", linewidth=2)
    axes[1, 0].set_title("Histogram")

    stats.probplot(resid, dist="norm", plot=axes[1, 1])
    axes[1, 1].set_title("Q-Q Plot")

    plt.suptitle(title, fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

    # --- Statistical tests ---
    print("=" * 60)
    print(f"DIAGNOSTIC REPORT: {title}")
    print("=" * 60)

    # Bias
    t_stat, t_p = stats.ttest_1samp(resid, 0)
    print(f"\n1. BIAS TEST (t-test for zero mean)")
    print(f"   Mean = {resid.mean():.4f}, p-value = {t_p:.4f}")
    print(f"   {'BIASED' if t_p < 0.05 else 'UNBIASED'}")

    # Autocorrelation
    lb = acorr_ljungbox(resid, lags=lb_lags, return_df=True)
    print(f"\n2. AUTOCORRELATION (Ljung-Box test)")
    for lag in lb_lags:
        p = lb.loc[lag, "lb_pvalue"]
        print(f"   Lag {lag:2d}: p-value = {p:.4f}  {'*** CORRELATED' if p < 0.05 else '    OK'}")

    # Normality
    sw_stat, sw_p = stats.shapiro(resid)
    print(f"\n3. NORMALITY (Shapiro-Wilk test)")
    print(f"   p-value = {sw_p:.4f}  {'NOT NORMAL' if sw_p < 0.05 else 'NORMAL'}")

    # Homoscedasticity
    n = len(resid)
    lev_stat, lev_p = stats.levene(resid.iloc[:n//2], resid.iloc[n//2:])
    print(f"\n4. CONSTANT VARIANCE (Levene's test)")
    print(f"   p-value = {lev_p:.4f}  {'HETEROSCEDASTIC' if lev_p < 0.05 else 'HOMOSCEDASTIC'}")
    print("=" * 60)


print("full_residual_check() function defined.")

In [ ]:
# Full report for naive method
full_residual_check(residuals_naive, title="Naive Method")

In [ ]:
# Full report for seasonal naive
full_residual_check(residuals_snaive, title="Seasonal Naive")

---
## 10. What to Do When Diagnostics Fail

| Problem | What it means | What to do |
|---|---|---|
| Autocorrelated residuals | Information left in data | Try a more complex model (ARIMA, ETS) |
| Non-zero mean | Systematic bias | Add a drift/intercept term |
| Increasing variance | Heteroscedasticity | Apply a log or Box-Cox transformation |
| Non-normal residuals | Prediction intervals unreliable | Use bootstrap intervals instead |

**The most important check is autocorrelation** — if residuals are correlated,
the model is leaving predictable information on the table, and a better
model exists.

In [ ]:
# Demonstrate: log-transforming the data before applying naive
log_passengers = np.log(passengers)
fitted_log_naive = log_passengers.shift(1)
residuals_log_naive = (log_passengers - fitted_log_naive).dropna()

full_residual_check(residuals_log_naive, title="Naive on Log-Transformed Passengers")

In [ ]:
print("After log-transforming, the heteroscedasticity is reduced.")
print("However, autocorrelation remains — we still need a better model")
print("(e.g., seasonal naive or ARIMA) to capture the seasonal pattern.")

---
## Summary

- **Residuals** $e_t = y_t - \hat{y}_t$ tell us whether a model has captured
  all the available signal.
- **Four diagnostic plots** (time plot, ACF, histogram, Q-Q) give a quick
  visual assessment.
- The **Ljung-Box test** formally tests for autocorrelation in residuals —
  significant results mean the model is leaving information on the table.
- **Bias** (non-zero mean), **heteroscedasticity** (changing variance), and
  **non-normality** are secondary but important for prediction intervals.
- The `check_residuals()` and `full_residual_check()` functions defined here
  can be reused throughout the remaining chapters.

In [ ]:
print("Next notebook: Evaluation Metrics — MAE, MSE, RMSE, MAPE, MASE")
print("and how to properly measure forecast accuracy.")